<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module2_Labs/Lab7_Parameter_Optimization_COBYLA_SPSA.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 7 — Parameter Optimization: COBYLA vs SPSA
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Understand why QAOA needs **classical parameter optimization**
2. Understand **COBYLA**: probe nearby points, build a linear model, take a step
3. Understand **SPSA**: estimate gradient with only 2 function evaluations regardless of dimension
4. Compare COBYLA vs SPSA on the 5-node Max-Cut problem
5. Understand **barren plateaus** — why optimization fails for large circuits
6. Explore **parameter initialization strategies**

---
### 📖 Background: Why Classical Optimization?

QAOA produces a state $|\varphi(\gamma,\beta)\rangle$ whose expected cut value $F(\gamma,\beta) = \langle H_C \rangle$ depends on the parameters. The quantum computer can **evaluate** $F$ but cannot **optimize** it directly.

The **classical optimizer** adjusts $\gamma$ and $\beta$ to maximize $F$.

For $p$ layers: **2p parameters** ($\gamma_1,\ldots,\gamma_p, \beta_1,\ldots,\beta_p$)

**COBYLA** (Constrained Optimization BY Linear Approximations):
- Probes $2d+1$ nearby points to estimate gradient ($d$ = number of parameters)
- Builds a linear local model, takes a step
- $O(d)$ evaluations per step — manageable for small $p$

**SPSA** (Simultaneous Perturbation Stochastic Approximation):
- Only **2 evaluations per step**, regardless of $d$
- Perturbs ALL parameters simultaneously in a random direction
- Noisier but far more efficient for large $p$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()

N_NODES = 5
EDGES   = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]
all_bitstrings = [format(i,'05b') for i in range(32)]

def cut_value(bs, edges):
    return sum(1 for u,v in edges if bs[u] != bs[v])

def build_qaoa_circuit(gammas, betas, n_qubits, edges, measure=True):
    p = len(gammas)
    qc = QuantumCircuit(n_qubits, n_qubits if measure else 0)
    qc.h(range(n_qubits))
    for layer in range(p):
        for u,v in edges:
            qc.cx(u,v); qc.rz(-gammas[layer],v); qc.cx(u,v)
        for i in range(n_qubits):
            qc.rx(2*betas[layer], i)
    if measure:
        qc.measure(range(n_qubits), range(n_qubits))
    return qc

def compute_F_exact(params, p):
    """Exact (statevector) expectation value — no shot noise."""
    gammas, betas = params[:p], params[p:]
    qc = build_qaoa_circuit(gammas, betas, N_NODES, EDGES, measure=False)
    sv = Statevector(qc)
    return sum(cut_value(bs,EDGES)*abs(amp)**2 for bs,amp in zip(all_bitstrings,sv.data))

def compute_F_shots(params, p, shots=1000):
    """Shot-based expectation value — has shot noise."""
    gammas, betas = params[:p], params[p:]
    qc = build_qaoa_circuit(gammas, betas, N_NODES, EDGES, measure=True)
    result = simulator.run(transpile(qc, simulator), shots=shots).result()
    counts = result.get_counts()
    total  = sum(counts.values())
    return sum(cut_value(bs,EDGES)*(cnt/total) for bs,cnt in counts.items())

print("Setup complete.")

---
## Part 1: COBYLA — Step-by-Step Walkthrough

In [ ]:
# ── 1.1  Manual COBYLA step to build intuition ────────────────────────────────
# p=1: 2 parameters (γ, β)
p = 1
gamma0, beta0 = 0.3, 0.6
d = 0.1  # step size

print("COBYLA Step-by-Step (p=1, γ₀=0.3, β₀=0.6):")
print("=" * 55)

# Evaluate at center and 4 neighbors
probes = [
    (gamma0,   beta0,   'center'),
    (gamma0+d, beta0,   'γ + δ'),
    (gamma0-d, beta0,   'γ - δ'),
    (gamma0,   beta0+d, 'β + δ'),
    (gamma0,   beta0-d, 'β - δ'),
]

print(f"\n{'Point':>10} | {'γ':>6} | {'β':>6} | {'F(γ,β)':>8}")
print("-" * 40)
results_probe = []
for gamma, beta, label in probes:
    F = compute_F_exact([gamma, beta], p=1)
    results_probe.append((gamma, beta, label, F))
    print(f"{label:>10} | {gamma:>6.3f} | {beta:>6.3f} | {F:>8.4f}")

print("\nCOBYLA learns from these evaluations:")
F_center = results_probe[0][3]
grad_gamma = (results_probe[1][3] - results_probe[2][3]) / (2*d)
grad_beta  = (results_probe[3][3] - results_probe[4][3]) / (2*d)
print(f"  ∂F/∂γ ≈ {grad_gamma:+.4f} → {'increase' if grad_gamma>0 else 'decrease'} γ")
print(f"  ∂F/∂β ≈ {grad_beta:+.4f} → {'increase' if grad_beta>0 else 'decrease'} β")

gamma_new = gamma0 + d * np.sign(grad_gamma)
beta_new  = beta0  + d * np.sign(grad_beta)
F_new = compute_F_exact([gamma_new, beta_new], p=1)
print(f"\nStep: γ: {gamma0:.3f} → {gamma_new:.3f}, β: {beta0:.3f} → {beta_new:.3f}")
print(f"F improvement: {F_center:.4f} → {F_new:.4f} (+{F_new-F_center:+.4f})")

In [ ]:
# ── 1.2  Full COBYLA run with scipy ──────────────────────────────────────────
p = 1
history_cobyla = []
eval_calls_cobyla = [0]

def objective_cobyla(params):
    F = compute_F_exact(params, p)
    history_cobyla.append(F)
    eval_calls_cobyla[0] += 1
    return -F

np.random.seed(7)
init_params = np.random.uniform(0, np.pi, 2*p)

result_cobyla = minimize(
    objective_cobyla, init_params,
    method='COBYLA',
    options={'maxiter': 200, 'rhobeg': 0.5}
)

best_cobyla = -result_cobyla.fun
print(f"COBYLA: F* = {best_cobyla:.4f} in {eval_calls_cobyla[0]} evaluations")
print(f"  γ* = {result_cobyla.x[0]:.4f} rad ({np.degrees(result_cobyla.x[0]):.1f}°)")
print(f"  β* = {result_cobyla.x[1]:.4f} rad ({np.degrees(result_cobyla.x[1]):.1f}°)")

---
## Part 2: SPSA — Only 2 Evaluations per Step

In [ ]:
# ── 2.1  SPSA manual demonstration ───────────────────────────────────────────
p = 1
gamma0, beta0 = 0.3, 0.6
delta = 0.2  # perturbation magnitude

# SPSA: pick ONE random direction ±δ_vec
np.random.seed(42)
delta_vec = delta * (2 * np.random.randint(0, 2, size=2) - 1)  # each element ±δ
params0   = np.array([gamma0, beta0])

F_plus  = compute_F_exact(params0 + delta_vec, p=1)
F_minus = compute_F_exact(params0 - delta_vec, p=1)
F_center = compute_F_exact(params0, p=1)

print("SPSA: only 2 evaluations to estimate gradient")
print(f"  Random direction δ_vec = {np.round(delta_vec,4)}")
print(f"  F(params + δ) = {F_plus:.4f}")
print(f"  F(params - δ) = {F_minus:.4f}")
print(f"  F(center)     = {F_center:.4f} (not used in SPSA!)")

# SPSA gradient estimate
grad_spsa = (F_plus - F_minus) / (2 * delta_vec)
print(f"\nSPSA gradient estimate:")
print(f"  ∂F/∂γ ≈ {grad_spsa[0]:+.4f}")
print(f"  ∂F/∂β ≈ {grad_spsa[1]:+.4f}")
print("\nKey: Only 2 evaluations regardless of how many parameters!")
print("COBYLA would need ~5 evaluations for 2 parameters, ~11 for 5 parameters, etc.")

In [ ]:
# ── 2.2  Full SPSA implementation ─────────────────────────────────────────────
def spsa_optimize(objective_fn, init_params, n_iter=200,
                  a=0.3, c=0.2, A=10, alpha=0.602, gamma_spsa=0.101):
    """
    SPSA optimizer.
    a, c:   initial step sizes for parameter update and gradient estimate
    A:      stability constant
    alpha, gamma_spsa: decay exponents
    """
    params  = np.array(init_params, dtype=float)
    history = []

    for k in range(1, n_iter+1):
        # Decaying step sizes
        a_k = a / (k + A)**alpha
        c_k = c / k**gamma_spsa

        # Random ±1 perturbation direction
        delta = 2*np.random.randint(0,2,size=len(params)) - 1

        # Two-point gradient estimate
        F_plus  = objective_fn(params + c_k*delta)
        F_minus = objective_fn(params - c_k*delta)
        grad    = (F_plus - F_minus) / (2*c_k*delta)   # sign: we want to MAXIMIZE

        # Update
        params += a_k * grad
        history.append(objective_fn(params))  # evaluation for tracking

    return params, history

p = 1
history_spsa = []

def objective_spsa(params):
    return compute_F_exact(params, p)

np.random.seed(7)
init_params = np.random.uniform(0, np.pi, 2*p)

best_params_spsa, history_spsa = spsa_optimize(
    objective_spsa, init_params, n_iter=150, a=0.5, c=0.3
)
best_spsa = compute_F_exact(best_params_spsa, p)

print(f"SPSA: F* = {best_spsa:.4f} using ~{3*150} evaluations (3 per iteration)")
print(f"  γ* = {best_params_spsa[0]:.4f} rad")
print(f"  β* = {best_params_spsa[1]:.4f} rad")

In [ ]:
# ── 2.3  Compare COBYLA vs SPSA convergence ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_cobyla, 'b-', alpha=0.8, lw=1.5, label=f'COBYLA: F*={best_cobyla:.3f}')
axes[0].plot(history_spsa,   'r-', alpha=0.8, lw=1.5, label=f'SPSA:   F*={best_spsa:.3f}')
axes[0].axhline(6, color='green', linestyle='--', lw=2, label='Optimal=6')
axes[0].axhline(3, color='gray',  linestyle=':',  lw=1, label='Random=3')
axes[0].set_xlabel('Function evaluations'); axes[0].set_ylabel('F(γ,β)')
axes[0].set_title('COBYLA vs SPSA Convergence (p=1)')
axes[0].legend(); axes[0].set_ylim(0,7)

# Running maximum (best found so far)
cobyla_best = np.maximum.accumulate(history_cobyla)
spsa_best   = np.maximum.accumulate(history_spsa)
axes[1].plot(cobyla_best, 'b-', lw=2, label='COBYLA best so far')
axes[1].plot(spsa_best,   'r-', lw=2, label='SPSA best so far')
axes[1].axhline(6, color='green', linestyle='--', lw=2, label='Optimal=6')
axes[1].set_xlabel('Function evaluations'); axes[1].set_ylabel('Best F found')
axes[1].set_title('Best F Found vs. Evaluations (p=1)')
axes[1].legend(); axes[1].set_ylim(0,7)

plt.tight_layout()
plt.show()

print("Comparison summary:")
print(f"  COBYLA: {len(history_cobyla)} evaluations, F* = {best_cobyla:.4f}")
print(f"  SPSA:   {len(history_spsa)} evaluations,   F* = {best_spsa:.4f}")
print("For p=1 (2 params), COBYLA is usually better.")
print("For p=5+ (10+ params), SPSA's efficiency advantage becomes critical.")

---
## Part 3: Barren Plateaus — Why Deep Circuits are Hard to Optimize

In [ ]:
# ── 3.1  Demonstrate gradient variance shrinks with depth ─────────────────────
# For random parameters, gradient magnitude decreases with circuit depth
# This is the 'barren plateau' phenomenon

print("Measuring gradient variance for different p layers...")
n_random_samples = 50
gradient_vars = []

for p in [1, 2, 3, 5]:
    grads = []
    for _ in range(n_random_samples):
        params = np.random.uniform(0, np.pi, 2*p)
        d_eps  = 0.01
        dp = np.zeros(2*p); dp[0] = d_eps  # perturb only γ_1
        F_plus  = compute_F_exact(params + dp, p)
        F_minus = compute_F_exact(params - dp, p)
        grad    = (F_plus - F_minus) / (2*d_eps)
        grads.append(grad)
    var_grad = np.var(grads)
    gradient_vars.append(var_grad)
    print(f"  p={p}: Var(∂F/∂γ₁) = {var_grad:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy([1,2,3,5], gradient_vars, 'bo-', ms=8, lw=2)
ax.set_xlabel('Number of QAOA layers (p)')
ax.set_ylabel('Gradient variance (log scale)')
ax.set_title('Barren Plateau: Gradient Variance Shrinks with Circuit Depth')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("\nAs p increases, gradients become exponentially small!")
print("This makes random initialization fail for deep circuits.")

---
### ✏️ Exercise 7.1 — COBYLA with Shot Noise

In a real quantum computer, every evaluation of $F$ is noisy (finite shots).

1. Run COBYLA optimization using `compute_F_shots` (shot-based) with **100 shots** and **1000 shots**
2. Compare convergence to the **exact** (statevector) result
3. What minimum shot count gives reliable optimization?
4. How does shot noise affect the final F* achieved?

In [ ]:
# YOUR CODE HERE
p = 1
all_histories = {}
all_best = {}

for shots_label, shots_val in [('Exact', None), ('1000 shots', 1000), ('100 shots', 100)]:
    hist = []

    def make_objective(s):
        def obj(params):
            if s is None:
                F = compute_F_exact(params, p)
            else:
                F = compute_F_shots(params, p, shots=s)
            hist.append(F)
            return -F
        return obj

    np.random.seed(7)
    init = np.random.uniform(0, np.pi, 2*p)
    res  = minimize(make_objective(shots_val), init, method='COBYLA',
                    options={'maxiter':150, 'rhobeg':0.5})
    best = -res.fun
    all_histories[shots_label] = hist
    all_best[shots_label] = best
    print(f"  {shots_label:12s}: F* = {best:.4f} ({len(hist)} evals)")

fig, ax = plt.subplots(figsize=(9, 4))
colors = {'Exact':'black', '1000 shots':'blue', '100 shots':'red'}
for label, hist in all_histories.items():
    smoothed = np.convolve(hist, np.ones(5)/5, mode='valid')  # smooth
    ax.plot(smoothed, color=colors[label], lw=1.5, alpha=0.8,
            label=f'{label}: F*={all_best[label]:.3f}')
ax.axhline(6, color='green', linestyle='--', lw=2, label='Optimal=6')
ax.set_xlabel('Evaluations'); ax.set_ylabel('F(γ,β) (smoothed)')
ax.set_title('COBYLA Convergence: Exact vs Shot-Based (p=1)')
ax.legend(); ax.set_ylim(0, 7)
plt.tight_layout()
plt.show()
print("\nConclusion: More shots → less noise → more reliable optimization.")
print("In practice, ~500-1000 shots is a reasonable trade-off.")

---
### ✏️ Exercise 7.2 — Parameter Initialization Strategies

Good initialization can significantly speed up QAOA optimization.

1. **Random**: initialize γ, β uniformly in [0, π]
2. **Warm start**: use the optimal p=1 solution as starting point for p=2
3. **Small angle**: initialize all γ, β near 0 (avoids barren plateaus)

For p=2, compare all three strategies in terms of:
- Final F* achieved
- Number of evaluations needed to reach F > 5.0

In [ ]:
# YOUR CODE HERE
p = 2

# Get p=1 optimal parameters (warm start seed)
np.random.seed(7)
init1 = np.random.uniform(0, np.pi, 2)
res1 = minimize(lambda p: -compute_F_exact(p, 1), init1, method='COBYLA',
                options={'maxiter':150,'rhobeg':0.5})
gamma1_opt, beta1_opt = res1.x
print(f"p=1 optimal: γ*={gamma1_opt:.3f}, β*={beta1_opt:.3f}, F*={-res1.fun:.3f}")

strategies = {
    'Random [0,π]':      np.random.uniform(0, np.pi, 4),
    'Warm start (p=1)':  np.array([gamma1_opt, gamma1_opt, beta1_opt, beta1_opt]),
    'Small angle [0,0.3]': np.random.uniform(0, 0.3, 4),
}

print(f"\nComparing p=2 initialization strategies:")
results_strat = {}

for strat_name, init_params in strategies.items():
    hist = []
    def obj(params, h=hist):
        F = compute_F_exact(params, p)
        h.append(F)
        return -F
    res = minimize(obj, init_params, method='COBYLA', options={'maxiter':200,'rhobeg':0.3})
    best_F = -res.fun
    evals_to_5 = next((i for i,f in enumerate(hist) if f>=5.0), len(hist))
    results_strat[strat_name] = {'hist':hist, 'best':best_F, 'evals_to_5':evals_to_5}
    print(f"  {strat_name:25s}: F*={best_F:.3f}, evals to F>5: {evals_to_5}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
colors = {'Random [0,π]':'red', 'Warm start (p=1)':'green', 'Small angle [0,0.3]':'blue'}
for name, data in results_strat.items():
    ax.plot(data['hist'], color=colors[name], lw=1.5, alpha=0.8,
            label=f"{name}: F*={data['best']:.3f}")
ax.axhline(6, color='black', linestyle='--', lw=2, label='Optimal=6')
ax.axhline(5, color='orange', linestyle=':', lw=1.5, label='Good threshold=5')
ax.set_xlabel('Evaluations'); ax.set_ylabel('F(γ,β)')
ax.set_title('p=2 QAOA: Initialization Strategy Comparison')
ax.legend(fontsize=9); ax.set_ylim(0, 7)
plt.tight_layout()
plt.show()

---
## ✅ Lab 7 Summary

| Optimizer | Evaluations/step | Best for | Weakness |
|-----------|-----------------|----------|----------|
| **COBYLA** | $O(2d+1)$ | Small $p$ (few params), low noise | Expensive for large $p$ |
| **SPSA** | 2 (always) | Large $p$, many params | Noisy gradient, slow convergence |

| Issue | Description | Fix |
|-------|-------------|-----|
| **Shot noise** | Finite shots → noisy $F$ estimates | Use ≥500 shots; SPSA handles noise better |
| **Barren plateaus** | Gradient exponentially small for deep circuits | Small-angle init, warm start from $p-1$ |
| **Local optima** | Many local maxima in the $F(\gamma,\beta)$ landscape | Multiple random restarts |

## 🔭 Preview of Lab 8
Next: **QAOA for Portfolio Optimization** — applying the same framework to a real financial problem using live stock data from Yahoo Finance.

---
*QOS Lab 7 | Prof. Chansu Yu | Cleveland State University*